# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logistic Regression gives me a simple starting model. Random Forest and Gradient Boosting let me test whether combinations of signals improve the ranking. I use the same outcome as Week 4: an impression drop of more than 20% from March to April.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'skills/README.md').exists())
OUTPUTS = ROOT / 'work/outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)
KEYS = ['client_hash_id', 'content_hash_id']
LABEL = 'is_declining_label'

def monthly_pages(month, start, end):
    # Week 4 already saved these page summaries. Rebuild only if a cache is absent.
    cache = OUTPUTS / f'w04_audit_{month}_v1.parquet'
    if cache.exists():
        return pd.read_parquet(cache)
    from huggingface_hub import get_token
    token = get_token()
    if not token:
        raise RuntimeError('A Hugging Face READ token is needed to rebuild the monthly cache.')
    con = duckdb.connect()
    con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [token])
    source = (f'hf://datasets/FlyRank/internship-warehouse/'
              f'fact_content_daily_performance/month={month}/*.parquet')
    frame = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
            COUNT(*) - COUNT(DISTINCT report_date) AS duplicate_rows,
            COUNT(DISTINCT report_date) FILTER (
                WHERE gsc_data_available IS TRUE) AS gsc_days,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND
                (gsc_impressions IS NULL OR gsc_clicks IS NULL OR gsc_sum_position IS NULL)
            ) AS missing_gsc_rows,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
            SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE AND
                report_date <= DATE '{start}' + INTERVAL 14 DAY) AS early15,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE AND
                report_date > DATE '{start}' + INTERVAL 14 DAY) AS late15
        FROM read_parquet('{source}')
        WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}'
        GROUP BY client_hash_id, content_hash_id
    """).df()
    assert frame['duplicate_rows'].sum() == 0
    frame.to_parquet(cache, index=False)
    return frame

march = monthly_pages('2026-03', '2026-03-02', '2026-03-31')
april = monthly_pages('2026-04', '2026-04-01', '2026-04-30')
assert march['duplicate_rows'].sum() == april['duplicate_rows'].sum() == 0
assert not march.duplicated(KEYS).any() and not april.duplicated(KEYS).any()

eligible = (march['gsc_days'].eq(30) & march['missing_gsc_rows'].eq(0)
            & march['impressions'].ge(100) & march['early15'].gt(0))
features = march.loc[eligible, KEYS +
    ['impressions', 'clicks', 'position_sum', 'early15', 'late15']].copy()
features = features.rename(columns={'impressions': 'impressions_feature_30d',
                                    'clicks': 'clicks_feature_30d'})
features['momentum_pct'] = 100 * (features['late15'] - features['early15']) / features['early15']
features['ctr_feature_30d_pct'] = 100 * features['clicks_feature_30d'] / features['impressions_feature_30d']
features['weighted_position_feature_30d'] = (
    features['position_sum'] / features['impressions_feature_30d']
).where(features['position_sum'].gt(0))

targets = april.loc[april['gsc_days'].eq(30) & april['missing_gsc_rows'].eq(0),
                    KEYS + ['impressions']].rename(columns={'impressions': 'target_impressions'})
model_frame = features.merge(targets, on=KEYS, validate='one_to_one')
model_frame[LABEL] = (model_frame['target_impressions']
                      < 0.80 * model_frame['impressions_feature_30d']).astype(int)
FEATURES = ['impressions_feature_30d', 'clicks_feature_30d',
            'ctr_feature_30d_pct', 'weighted_position_feature_30d', 'momentum_pct']
model_frame = model_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + [LABEL]).copy()
assert not model_frame.duplicated(KEYS).any()
print(f'March eligible pages: {len(features):,}')
print(f'Pages with a complete April outcome and five usable inputs: {len(model_frame):,}')
print(f'Observed April decline rate: {model_frame[LABEL].mean():.1%}')

assert LABEL not in FEATURES and 'target_impressions' not in FEATURES
assert pd.Timestamp('2026-03-31') < pd.Timestamp('2026-04-01')


March eligible pages: 62,397
Pages with a complete April outcome and five usable inputs: 50,641
Observed April decline rate: 47.3%


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Pages from the same client can share tracking or site-wide changes, so I keep each client's pages together. I reserve a quarter of clients for testing and choose the model using three grouped folds within the training data. Seed 7 was chosen using group sizes to keep the test set close to a quarter of the pages.

This split checks performance on other clients within one period. The difference in decline rates between groups suggests that client mix matters; later months still need testing.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
train_idx, test_idx = next(splitter.split(model_frame, groups=model_frame['client_hash_id']))
train = model_frame.iloc[train_idx].copy()
test = model_frame.iloc[test_idx].copy()
assert set(train['client_hash_id']).isdisjoint(test['client_hash_id'])
assert train[LABEL].nunique() == test[LABEL].nunique() == 2
print(f'Train: {len(train):,} pages from {train.client_hash_id.nunique()} clients; '
      f'decline rate {train[LABEL].mean():.1%}')
print(f'Test: {len(test):,} pages from {test.client_hash_id.nunique()} clients; '
      f'decline rate {test[LABEL].mean():.1%}')


Train: 37,081 pages from 24 clients; decline rate 49.6%
Test: 13,560 pages from 8 clients; decline rate 41.3%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Precision@50 matches the team's 50-page review queue. Recalculating the Week 4 rule on the same test pages keeps the comparison fair. Checking smaller queues shows whether the result depends on how many pages the team can review.

In [3]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def model_inputs(frame):
    x = frame[FEATURES].copy()
    # March counts have long tails; cap very large momentum gains from tiny early totals.
    x['impressions_feature_30d'] = np.log1p(x['impressions_feature_30d'])
    x['clicks_feature_30d'] = np.log1p(x['clicks_feature_30d'])
    x['momentum_pct'] = x['momentum_pct'].clip(upper=200)
    return x

def rank_pages(frame, scores):
    return frame.assign(score=scores).sort_values(
        ['score', 'impressions_feature_30d'],
        ascending=[False, False], kind='stable')

def precision_at_k(ranked, k):
    return ranked[LABEL].head(k).mean()

candidates = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=50,
        random_state=7, n_jobs=-1),
    'Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=150, max_leaf_nodes=15, min_samples_leaf=80,
        learning_rate=0.05, l2_regularization=1, random_state=7),
}

# Choose the model using training clients only.
fold_rows = []
folds = GroupKFold(n_splits=3)
for fold, (fit_idx, valid_idx) in enumerate(
        folds.split(train, groups=train['client_hash_id']), start=1):
    fit = train.iloc[fit_idx]
    valid = train.iloc[valid_idx]
    assert set(fit['client_hash_id']).isdisjoint(valid['client_hash_id'])
    for name, candidate in candidates.items():
        fitted = clone(candidate).fit(model_inputs(fit), fit[LABEL])
        scores = fitted.predict_proba(model_inputs(valid))[:, 1]
        fold_rows.append({'model': name, 'fold': fold,
                          'Precision@50': precision_at_k(rank_pages(valid, scores), 50)})

validation = pd.DataFrame(fold_rows).pivot(index='model', columns='fold', values='Precision@50')
validation.columns = [f'Fold {fold}' for fold in validation.columns]
validation['Mean'] = validation.mean(axis=1)
display(validation.round(3))
selected_name = validation['Mean'].idxmax()
print('Selected using training-client folds:', selected_name)

# Refit on all training clients, then use the same test pages for every method.
model = clone(candidates[selected_name]).fit(model_inputs(train), train[LABEL])
logit = clone(candidates['Logistic Regression']).fit(model_inputs(train), train[LABEL])
test['model_score'] = model.predict_proba(model_inputs(test))[:, 1]
test['logit_score'] = logit.predict_proba(model_inputs(test))[:, 1]
match = test['impressions_feature_30d'].ge(500) & test['momentum_pct'].le(-20)
test['baseline_score'] = (-test['momentum_pct']).where(match, 0)
assert match.sum() >= 50

rankings = {
    'Week 4 rule': rank_pages(test, test['baseline_score']),
    'Logistic Regression': rank_pages(test, test['logit_score']),
    selected_name: rank_pages(test, test['model_score']),
}
comparison = pd.DataFrame([
    {'Method': name, 'Precision@10': precision_at_k(ranked, 10),
     'Precision@20': precision_at_k(ranked, 20),
     'Precision@50': precision_at_k(ranked, 50),
     'Test decline rate': test[LABEL].mean()}
    for name, ranked in rankings.items()
])
display(comparison.round(3))
print(f'Week 4 rule candidates in test: {match.sum():,}')
model_ranked = rankings[selected_name]


,Fold 1,Fold 2,Fold 3,Mean
model,,,,
Gradient Boosting,0.94,0.94,0.88,0.920
Logistic Regression,0.88,0.92,0.84,0.880
Random Forest,1.00,0.92,0.92,0.947


Selected using training-client folds: Random Forest


,Method,Precision@10,Precision@20,Precision@50,Test decline rate
0,Week 4 rule,0.9,0.85,0.86,0.413
1,Logistic Regression,0.9,0.95,0.88,0.413
2,Random Forest,1.0,0.95,0.90,0.413


Week 4 rule candidates in test: 3,984


Random Forest's small lead makes it worth testing on a later period before replacing the rule. I have already inspected this test split during development, so I treat this comparison as exploratory.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# Importance measures how much each input helped the forest separate the labels.
importance = pd.Series(model.feature_importances_, index=FEATURES, name='split_importance')
display(importance.sort_values(ascending=False).round(3).to_frame())

top50 = model_ranked.head(50)
wrong_top50 = top50.loc[top50[LABEL].eq(0)]
missed = model_ranked.iloc[50:].loc[model_ranked.iloc[50:][LABEL].eq(1)]
print(f'Random Forest top 50 with no April decline: {len(wrong_top50)} of 50')
print(f'Declining test pages outside the Random Forest top 50: {len(missed):,}')

# Show examples as ranges so the notebook does not reveal exact page measurements.
cases = pd.concat([wrong_top50.head(3).assign(case='Top 50, no decline'),
                   missed.head(3).assign(case='Declined, outside top 50')]).copy()
cases['March volume'] = pd.cut(cases['impressions_feature_30d'],
    [99, 499, 2999, 29999, np.inf],
    labels=['100-499', '500-2,999', '3,000-29,999', '30,000+'])
cases['March momentum'] = pd.cut(cases['momentum_pct'],
    [-np.inf, -50, -20, 0, 20, np.inf],
    labels=['loss >50%', 'loss 20-50%', 'loss 0-20%', 'gain 0-20%', 'gain >20%'])
cases['Position tier'] = pd.cut(cases['weighted_position_feature_30d'],
    [0, 3, 10, 20, np.inf],
    labels=['0-3', '3-10', '10-20', '20+'])
display(cases[['case', 'March volume', 'March momentum', 'Position tier']].reset_index(drop=True))


,split_importance
momentum_pct,0.448
impressions_feature_30d,0.194
ctr_feature_30d_pct,0.178
weighted_position_feature_30d,0.099
clicks_feature_30d,0.081


Random Forest top 50 with no April decline: 5 of 50
Declining test pages outside the Random Forest top 50: 5,552


,case,March volume,March momentum,Position tier
0,"Top 50, no decline","3,000-29,999",loss >50%,3-10
1,"Top 50, no decline","30,000+",loss >50%,3-10
2,"Top 50, no decline","3,000-29,999",loss >50%,20+
3,"Declined, outside top 50","3,000-29,999",loss >50%,20+
4,"Declined, outside top 50","3,000-29,999",loss >50%,20+
5,"Declined, outside top 50","3,000-29,999",loss >50%,20+


The importance pattern fits the rule's use of recent momentum loss. Volume and CTR may help distinguish pages with similar losses, although related inputs can share importance.

Similar March losses appear among both false alarms and missed declines. A temporary dip may recover, while a lasting decline can still fall below the review cutoff. I would check demand, tracking, and page context before deciding whether a refresh is useful.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.